In [1]:
import sys
sys.path.append("..")
import pandas as pd
import lightgbm as lgb
from helpers import load, feature_names, swap_sides, organic_splits

organic, farm = load()
names = feature_names()
train, val, test = organic_splits(organic)

In [4]:
BEST_PARAMS = {    "objective": "cross_entropy",   # accepts soft labels like 0.75 (plain "binary" does not)
    "learning_rate": 0.05,
    "num_leaves": 15,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.8,
    "feature_pre_filter": False,    # lets you change min_data_in_leaf between runs
    "verbose": -1,
    "seed": 0,}     # paste PARAMS from Notebook 2, after your tuning
BEST_TREES = 527        # paste model.best_iteration from Notebook 2's Cell 11

trainval = pd.concat([train, val])
farm_rows = farm[~(farm.aOriginalPlayerId.isin(set(test.aOriginalPlayerId))
                   | farm.bOriginalPlayerId.isin(set(test.aOriginalPlayerId)))]
both = pd.concat([trainval.assign(weight=1.0),
                  farm_rows.assign(weight=farm_rows.repeats / 4)], ignore_index=True)
table = pd.concat([both, swap_sides(both, names)], ignore_index=True)

final = lgb.train(BEST_PARAMS,
                  lgb.Dataset(table[names], label=table.label, weight=table.weight, feature_name=names),
                  num_boost_round=int(BEST_TREES * 1.1))
final.save_model("../models/fight.txt")

In [5]:
!cd .. && uv run python export_model.py models/fight.txt data/fights-v27.csv \
    --out ../src/bot/ml/models/fight-v27.json \
    --parity ../test/fixtures/fightModelParity-v27.json \
    --note "test logloss 0.xx, AUC 0.xx"

wrote ../src/bot/ml/models/fight-v27.json (717 KB, 579 trees)
wrote parity fixture ../test/fixtures/fightModelParity-v27.json (200 rows)


In [6]:
!cd ../.. && npx jest test/fightModel.test.ts 2>&1 | tail -15

(node:810) ExperimentalWarning: Type Stripping is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
 PASS  test/fightModel.test.ts
  fight model runtime
    ✓ reproduces LightGBM's predictions exactly (7 ms)
    ✓ is symmetric: P(A beats B) + P(B beats A) = 1 (1 ms)
    ✓ refuses a model trained on a different feature spec (6 ms)
    ✓ committed model fight-v27.json matches its Python parity fixture (89 ms)

Test Suites: 1 passed, 1 total
Tests:       4 passed, 4 total
Snapshots:   0 total
Time:        1.113 s
Ran all test suites matching test/fightModel.test.ts.
